# Data Pre-processing: Deteksi Kecanduan Smartphone
**Tujuan:** Membersihkan data, melakukan transformasi (encoding & scaling), dan menangani ketidakseimbangan kelas (*class imbalance*) menggunakan SMOTE agar dataset siap dilatih oleh algoritma *Machine Learning*.

## 1. Feature Selection (Penghapusan Fitur yang Tidak Relevan)
Menghapus kolom ID pengguna karena tidak memiliki pola prediktif, serta menghapus `addiction_level` karena target klasifikasi kita adalah `addicted_label` (menghindari kebocoran data / *target leakage*).

In [4]:
# ==========================================
# 1. IMPORT LIBRARY & LOAD DATA
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Load raw dataset
file_path = 'Smartphone_Usage_And_Addiction_Analysis_7500_Rows.csv'
df = pd.read_csv(file_path)

print("Dataset Original berhasil dimuat!")
print(f"Dimensi awal: {df.shape[0]} Baris, {df.shape[1]} Kolom")

ModuleNotFoundError: No module named 'imblearn'

## 2. Encoding Data Kategorikal
Algoritma *Machine Learning* hanya dapat memproses data numerik.
* **Label Mapping:** Untuk fitur `academic_work_impact` (Yes/No) dan `stress_level` (Low/Medium/High).
* **One-Hot Encoding:** Untuk fitur `gender` karena bersifat nominal.

In [ ]:
# ==========================================
# 2. FEATURE SELECTION (DROP COLUMNS)
# ==========================================
kolom_drop = ['transaction_id', 'user_id', 'addiction_level']
df.drop(columns=kolom_drop, inplace=True, errors='ignore')

print("Kolom yang tersisa setelah di-drop:\n", df.columns.tolist())

## 3. Data Splitting (Pembagian Train & Test Set)
Membagi data menjadi **80% Data Latih** dan **20% Data Uji**. Penggunaan `stratify=y` sangat krusial agar proporsi kelas target (70:30) tetap sama pada kedua set data.

In [ ]:
# ==========================================
# 3. ENCODING KATEGORIKAL
# ==========================================
# Mapping Biner & Ordinal
df['academic_work_impact'] = df['academic_work_impact'].map({'No': 0, 'Yes': 1})
df['stress_level'] = df['stress_level'].map({'Low': 0, 'Medium': 1, 'High': 2})

# One-Hot Encoding
df = pd.get_dummies(df, columns=['gender'], dtype=int)

print("Data setelah Encoding (5 baris pertama):")
display(df.head(3))

## 4. Feature Scaling (Standardisasi)
Menyamakan skala fitur numerik. *Scaling* di-fit secara eksklusif hanya pada `X_train` untuk mencegah *Data Leakage* dari `X_test`.

In [ ]:
# ==========================================
# 4. DATA SPLITTING
# ==========================================
X = df.drop(columns=['addicted_label'])
y = df['addicted_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Dimensi X_train: {X_train.shape}")
print(f"Dimensi X_test: {X_test.shape}")

## 5. SMOTE (Synthetic Minority Over-sampling Technique)
Menangani ketidakseimbangan data target dengan membuat sampel sintesis untuk kelas minoritas ("Tidak Kecanduan"). **Perhatian:** SMOTE hanya diaplikasikan pada data latih (`X_train`), tidak pada data uji.

In [ ]:
# ==========================================
# 5. FEATURE SCALING (STANDARD SCALER)
# ==========================================
scaler = StandardScaler()

kolom_numerik = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'app_opens_per_day', 'weekend_screen_time']

# Fit dan Transform pada Data Latih
X_train[kolom_numerik] = scaler.fit_transform(X_train[kolom_numerik])

# Transform HANYA pada Data Uji (tanpa melakukan fit)
X_test[kolom_numerik] = scaler.transform(X_test[kolom_numerik])

print("Scaling berhasil diterapkan pada fitur numerik.")

## 6. Export Dataset (Menyimpan Hasil Pre-processing)
Menyimpan data latih (yang sudah di-SMOTE) dan data uji ke dalam format CSV agar bisa digunakan pada *notebook* pemodelan (Random Forest & XGBoost).

In [ ]:
# ==========================================
# 6. HANDLING IMBALANCED DATA (SMOTE)
# ==========================================
print("--- Distribusi Target Sebelum SMOTE (Data Latih) ---")
print(y_train.value_counts(normalize=True) * 100)

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\n--- Distribusi Target Setelah SMOTE (Data Latih) ---")
print(y_train_smote.value_counts(normalize=True) * 100)
print(f"\nDimensi X_train_smote sekarang: {X_train_smote.shape}")

In [ ]:
# ==========================================
# 7. EXPORT PRE-PROCESSED DATA
# ==========================================
# Gabungkan kembali X dan y untuk disimpan
train_smote_df = pd.concat([X_train_smote, y_train_smote], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

# Simpan menjadi CSV
train_smote_df.to_csv('train_preprocessed_smote.csv', index=False)
test_df.to_csv('test_preprocessed.csv', index=False)

print("Data berhasil diekspor menjadi:")
print("1. 'train_preprocessed_smote.csv' (Data latih untuk training model)")
print("2. 'test_preprocessed.csv' (Data uji untuk evaluasi model)")